# Кибериммунный подход к разработке. Учебный пример "Светофор"

## Об авторе 

Этот блокнот разработан для вас Сергеем Соболевым, sergey.p.sobolev@kaspersky.com

Больше информации о кибериммунном подходе можно найти на странице https://github.com/sergey-sobolev/cyberimmune-systems/wiki/%D0%9A%D0%B8%D0%B1%D0%B5%D1%80%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82

Подписывайтесь на телеграм-канал @learning_cyberimmunity (https://t.me/learning_cyberimmunity)

Обучающие видео на тему кибериммунного подхода вы можете найти на youtube канале https://www.youtube.com/@learning_cyberimmunity/

## О примере 

Светофор - это, на первый взгляд, очень простая система, но она оказывает критическое влияние на безопасность дорожного движения. 
Применим идеи конструктивной безопасности к архитектуре и реализации прототипа светофора. За отправную точку возьмём [код базового примера](https://colab.research.google.com/github/sergey-sobolev/cyberimmune-systems-basic-demo-notebook01/blob/main/cyberimmunity-basics.ipynb) и немного его переработаем.

Для переработки будем использовать описание архитектуры решения, которое обсуждалось на занятии.

## Простая реализация выбранной политики архитектуры

Будем использовать политику архитектуры, показанную на рис. 1.

![Рис. 1. Политика архитектуры](images/tl-archpol-0.02.png)

Рис. 1. Политика архитектуры светофора

1. Создадим функциональные компоненты (сущности 1-4) и монитор безопасности, который будет контролировать их взаимодействие, в том числе реализовывать контроль конфигураций светофора (сущность №5 на архитектурной диаграмме)
2. Определим политики безопасности
3. Сымитируем запрос на изменение режима для проверки работы всех элементов

- В качестве интерфейса взаимодействия используем очереди сообщений, у каждой сущности есть своя «персональная» очередь, ассоциированная с ней
- Компоненты 1-4 отправляют сообщения только в очередь monitor сущности SecurityMonitor
- SecurityMonitor проверяет сообщения на соответствие политикам безопасности, в случае положительного решения перенаправляет сообщение в очередь соответствующей сущности

В коде назовём сущности следующим образом
1. Связь - CitySystemConnector
2. Система управления светофора - ControlSystem
3. Управление светодиодами - LightsGPIO
4. Система диагностики - SelfDiagnosticsSystem

Логику контроля режимов светофора (компонент №5 на рис. 1) реализуем в виде политик безопасности в мониторе безопасности

![Рис. 2. Политика архитектуры с именами классов](images/tl-archpol-code.png)

Рис. 2. Политика архитектуры с именами классов

Очередь событий для монитора безопасности: все запросы от сущностей друг к другу должны отправляться только в неё

In [1]:
from multiprocessing import Queue
monitor_events_queue = Queue()

Зафиксируем формат сообщений

In [2]:
from multiprocessing import Queue, Process
from multiprocessing.queues import Empty
from dataclasses import dataclass
import json
from time import sleep

@dataclass
class Event:
    source: str       # отправитель
    destination: str  # получатель
    operation: str    # запрашиваемое действие
    parameters: str   # параметры (JSON строка)

@dataclass
class ControlEvent:
    operation: str

monitor_events_queue = Queue()

# ЗАДАНИЕ: Добавлены конфигурации с секцией "left_arrow"
traffic_lights_allowed_configurations = [
    {"direction_1": "red", "direction_2": "green", "left_arrow": "off"},
    {"direction_1": "red", "direction_2": "red", "left_arrow": "off"},    
    {"direction_1": "yellow", "direction_2": "yellow", "left_arrow": "off"},    
    {"direction_1": "off", "direction_2": "off", "left_arrow": "off"},
    {"direction_1": "green", "direction_2": "red", "left_arrow": "on"}, # Разрешен поворот
]

### Монитор безопасности

Ниже в методе _check_policies можно увидеть пример политики безопасности:

```python
if event.source == "ControlSystem" \
        and event.destination == "LightsGPIO" \
        and event.operation == "set_mode" \ 
        and self._check_mode(event.operation):
    authorized = True
```            

в этом примере проверяется отправитель сообщения, получатель, запрашиваемая операция и даже параметры операции. Это максимально жёсткий вариант, очевидно, в зависимости от ситуации количество проверок можно уменьшить.

А пока это место для экспериментов, как можно из монитора безопасности заблокировать взаимодействие между сущностями.

In [3]:
class Monitor(Process):
    def __init__(self, events_q: Queue):
        super().__init__()
        self._events_q = events_q
        self._control_q = Queue()
        self._entity_queues = {}
        self._force_quit = False

    def add_entity_queue(self, entity_id: str, queue: Queue):
        print(f"[монитор] регистрация сущности: {entity_id}")
        self._entity_queues[entity_id] = queue

    def _check_mode(self, mode_str: str) -> bool:
        try:
            mode = json.loads(mode_str)
            return mode in traffic_lights_allowed_configurations
        except:
            return False

    def _check_policies(self, event):
        authorized = False
        if not isinstance(event, Event): return False

        # Политика 1: Управление светофором (ControlSystem -> LightsGPIO)
        if event.source == "ControlSystem" and event.destination == "LightsGPIO" \
                and event.operation == "set_mode" and self._check_mode(event.parameters):
            authorized = True

        # ЗАДАНИЕ: Политика 2: Самодиагностика (LightsGPIO -> SelfDiagnosticsSystem)
        if event.source == "LightsGPIO" and event.destination == "SelfDiagnosticsSystem" \
                and event.operation == "report_status":
            authorized = True

        return authorized

    def _proceed(self, event):
        try:
            dst_q: Queue = self._entity_queues[event.destination]
            dst_q.put(event)
        except Exception as e:
            print(f"[монитор] ошибка пересылки: {e}")

    def run(self):
        print('[монитор] запущен')
        while not self._force_quit:
            try:
                event = self._events_q.get(timeout=0.1)
                if self._check_policies(event):
                    self._proceed(event)
            except Empty:
                pass
            self._check_control_q()

    def stop(self):
        self._control_q.put(ControlEvent(operation='stop'))

    def _check_control_q(self):
        try:
            if self._control_q.get_nowait().operation == 'stop':
                self._force_quit = True
        except Empty:
            pass

### Сущность ControlSystem

Эта сущность отправляет сообщение для другой сущности (LightsGPIO)

In [4]:
class ControlSystem(Process):
    def __init__(self, monitor_queue: Queue):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()

    def entity_queue(self): return self._own_queue

    def run(self):        
        print(f'[{self.__class__.__name__}] запуск цикла управления')
        # ЗАДАНИЕ: Работаем произвольное время (цикл из 3 переключений)
        for _ in range(3): 
            mode = {"direction_1": "red", "direction_2": "green", "left_arrow": "off"}
            event = Event(source=self.__class__.__name__, destination='LightsGPIO', 
                          operation='set_mode', parameters=json.dumps(mode))
            self.monitor_queue.put(event)
            sleep(3) 

class LightsGPIO(Process):
    def __init__(self, monitor_queue: Queue):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()

    def entity_queue(self): return self._own_queue

    def run(self):        
        print(f'[{self.__class__.__name__}] ожидание команд...')
        while True:
            try:
                # Работаем, пока приходят сообщения (таймаут 5 сек)
                event = self._own_queue.get(timeout=5.0)
                if event.operation == "set_mode":
                    print(f"[{self.__class__.__name__}] Применен режим: {event.parameters}")
                    
                    # ЗАДАНИЕ: Отправка отчета в систему диагностики
                    report = Event(source=self.__class__.__name__, destination="SelfDiagnosticsSystem",
                                   operation="report_status", parameters="Hardware OK")
                    self.monitor_queue.put(report)
            except Empty:
                print(f"[{self.__class__.__name__}] Команд нет, завершение")
                break

# ЗАДАНИЕ: Реализация сущности самодиагностики
class SelfDiagnosticsSystem(Process):
    def __init__(self, monitor_queue: Queue):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()

    def entity_queue(self): return self._own_queue

    def run(self):
        print(f'[{self.__class__.__name__}] система диагностики готова')
        while True:
            try:
                event = self._own_queue.get(timeout=10.0)
                if event.operation == "report_status":
                    print(f"[SelfDiagnostics] СООБЩЕНИЕ: {event.source} статус -> {event.parameters}")
            except Empty:
                break

### Сущность LightsGPIO

Эта сущность ждёт входящее сообщение в течение заданного периода времени, если получает - обрабатывает и завершает работу или выходит по таймауту.

In [5]:
from multiprocessing import Queue, Process
from time import sleep


class LightsGPIO(Process):

    def __init__(self, monitor_queue: Queue):
        # вызываем конструктор базового класса
        super().__init__()
        # мы знаем только очередь монитора безопасности для взаимодействия с другими сущностями
        # прямая отправка сообщений в другую сущность запрещена в концепции FLASK
        self.monitor_queue = monitor_queue
        # создаём собственную очередь, в которую монитор сможет положить сообщения для этой сущности
        self._own_queue = Queue()

    def entity_queue(self):
        return self._own_queue

    # основной код сущности
    def run(self):        
        print(f'[{self.__class__.__name__}] старт')
        attempts = 10
        while attempts > 0:
            try:
                event: Event = self._own_queue.get_nowait()
                if event.operation == "set_mode":
                    print(f"[{self.__class__.__name__}] {event.source} запрашивает изменение режима {event.parameters}")
                    print(f"[{self.__class__.__name__}] новый режим: {event.parameters}!")
                    break
            except Empty:
                sleep(0.2)
                attempts -= 1
        print(f'[{self.__class__.__name__}] завершение работы')

In [6]:
class SelfDiagnosticsSystem(Process):
    def __init__(self, monitor_queue: Queue):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()

    def entity_queue(self):
        return self._own_queue

    def run(self):
        print(f'[{self.__class__.__name__}] старт системы диагностики')
        while True:
            try:
                event: Event = self._own_queue.get(timeout=1.0)
                if event.operation == "report_status":
                    print(f"[ДИАГНОСТИКА] Получен статус от {event.source}: {event.parameters}")
            except Empty:
                continue

### Инициализируем монитор и сущности

In [7]:
monitor = Monitor(monitor_events_queue)
control_system = ControlSystem(monitor_events_queue)
lights_gpio = LightsGPIO(monitor_events_queue)
self_diag = SelfDiagnosticsSystem(monitor_events_queue)

monitor.add_entity_queue("ControlSystem", control_system.entity_queue())
monitor.add_entity_queue("LightsGPIO", lights_gpio.entity_queue())
monitor.add_entity_queue("SelfDiagnosticsSystem", self_diag.entity_queue())

monitor.start()
self_diag.start()
lights_gpio.start()
control_system.start()

# Ожидание и завершение
sleep(7)
monitor.stop()
monitor.join()

[монитор] регистрация сущности: ControlSystem
[монитор] регистрация сущности: LightsGPIO
[монитор] регистрация сущности: SelfDiagnosticsSystem
[монитор] запущен
[SelfDiagnosticsSystem] старт системы диагностики
[LightsGPIO] старт
[ControlSystem] запуск цикла управления
[LightsGPIO] ControlSystem запрашивает изменение режима {"direction_1": "red", "direction_2": "green", "left_arrow": "off"}
[LightsGPIO] новый режим: {"direction_1": "red", "direction_2": "green", "left_arrow": "off"}!
[LightsGPIO] завершение работы


регистрируем очереди сущностей в мониторе

In [8]:
monitor.add_entity_queue(control_system.__class__.__name__, control_system.entity_queue())
monitor.add_entity_queue(lights_gpio.__class__.__name__, lights_gpio.entity_queue())

[монитор] регистрация сущности: ControlSystem
[монитор] регистрация сущности: LightsGPIO


### Запускаем всё

Ожидаемая последовательность событий

![Диаграмма последовательности вызовов](https://www.plantuml.com/plantuml/png/dPBVIiCm6CNlynIvdtk1NSZ02n4KXJr1w886-cUqcR0xgoBpIdoJLgqhtRg-mfStygJPM3js9OLyoPTpVia97ITQn7eU-6o6gZmr4w7c5r6euyYVB18j0ouIxhb6JpIHtZnMUd4JXKf7iPK5RjgJNQlx1vrStbtTMeNVhXZR0VdmV6yQ34QSQjhI5sNcWrE5QMrUgQHlysAUq7oZqcxyq1hbW6KxG8S5KWEBPHMe5MLeOBa6uHd4YWbVSs1JQg1OKktEF3ATSTI2Vk7Os6b6AupGerctn3uM5WWfn_OAxGRGr4OogTtktjCzmt3utyZ2q-fHQBb_JrSEv177oHigRB1E2ChOL1vxfPz8ZWjdBhv9ELn5Bw_vfFf4L2fFlZsEtkBBpNjxWH8mkyHWbbZbC1TCXbCsne1Vxmy0)

In [9]:
# Инициализация объектов
monitor = Monitor(monitor_events_queue)
control_system = ControlSystem(monitor_events_queue)
lights_gpio = LightsGPIO(monitor_events_queue)
self_diag = SelfDiagnosticsSystem(monitor_events_queue)

# Регистрация сущностей в мониторе
monitor.add_entity_queue("ControlSystem", control_system.entity_queue())
monitor.add_entity_queue("LightsGPIO", lights_gpio.entity_queue())
monitor.add_entity_queue("SelfDiagnosticsSystem", self_diag.entity_queue())

# Старт процессов
monitor.start()
self_diag.start()
lights_gpio.start()
control_system.start()

# Даем системе поработать 10 секунд
sleep(10)

# Корректная остановка монитора
monitor.stop()
monitor.join()

print("Тест завершен.")

[монитор] регистрация сущности: ControlSystem
[монитор] регистрация сущности: LightsGPIO
[монитор] регистрация сущности: SelfDiagnosticsSystem
[монитор] запущен
[SelfDiagnosticsSystem] старт системы диагностики
[LightsGPIO] старт
[ControlSystem] запуск цикла управления
[LightsGPIO] ControlSystem запрашивает изменение режима {"direction_1": "red", "direction_2": "green", "left_arrow": "off"}
[LightsGPIO] новый режим: {"direction_1": "red", "direction_2": "green", "left_arrow": "off"}!
[LightsGPIO] завершение работы
Тест завершен.


### Теперь останавливаем

In [10]:
# Создаем объекты
monitor = Monitor(monitor_events_queue)
control_system = ControlSystem(monitor_events_queue)
lights_gpio = LightsGPIO(monitor_events_queue)
self_diag = SelfDiagnosticsSystem(monitor_events_queue) # Создаем диагностику

# РЕГИСТРАЦИЯ: обязательно добавляем очереди в монитор
monitor.add_entity_queue("ControlSystem", control_system.entity_queue())
monitor.add_entity_queue("LightsGPIO", lights_gpio.entity_queue())
monitor.add_entity_queue("SelfDiagnosticsSystem", self_diag.entity_queue()) # Регистрируем диагностику

# Запуск всех процессов
monitor.start()
self_diag.start()
lights_gpio.start()
control_system.start()

[монитор] регистрация сущности: ControlSystem
[монитор] регистрация сущности: LightsGPIO
[монитор] регистрация сущности: SelfDiagnosticsSystem
[монитор] запущен
[SelfDiagnosticsSystem] старт системы диагностики
[LightsGPIO] старт
[ControlSystem] запуск цикла управления
[LightsGPIO] ControlSystem запрашивает изменение режима {"direction_1": "red", "direction_2": "green", "left_arrow": "off"}
[LightsGPIO] новый режим: {"direction_1": "red", "direction_2": "green", "left_arrow": "off"}!
[LightsGPIO] завершение работы


## Заключение

В этом блокноте продемонстрирован базовый функционал контролируемого изменения режима работы светофора. 

В примере не реализованы некоторые сущности и большая часть логики работы светофора, которую можно предположить по архитектурной диаграмме. Попробуйте сделать это самостоятельно!

## Упражнения

Уровень "Новичок"

- в коде ControlSystem измените режим на недопустимый (два зелёных) и выполните все ячейки. Убедитесь, что монитор безопасности заблокировал сообщение, как нарушающее политику безопасности
- измените политики безопасности так, чтобы был возможен режим "моргающий жёлтый" (yellow_blinking), переводящий перекрёсток в режим нерегулируемого

Уровень "Средней сложности"

- добавьте политики безопасности для доп. секций со стрелками (поворот налево или направо)
- измените код сущностей, чтобы они не завершали работу после одного сообщения, а работали произвольное время (см. реализацию монитора безопасности)
- реализуйте сущность само-диагностики (SelfDiagnostics) и отправку сообщений от LightsGPIO (необходимо доработать политики безопасности!)

Уровень "Продвинутый"

- измените код сущности ControlSystem, реализуйте смену режимов по таймеру (заданную длительность зелёного по каждому направлению)
- реализуйте сущность CitySystemConnector, которая будет имитировать получение изменения режима, реализуйте взаимодействие CitySystemConnector и ControlSystem (понадобится доработать политики безопасности). Например, изменение длительности зелёного по направлениям или отключение светофора (перевод перекрёстка в режим нерегулируемого)
- реализуйте передачу в компонент CitySystemConnector информации об исправности светофора (статус самодиагностики; текущий режим работы). В компоненте реализуйте вывод в виде сообщений о состоянии системы